# Backtest de TrendWatch

**Esta práctica usa datos sintéticos reproducibles**, no productos realmente virales. Sirve para comprobar el detector de picos (*z-score*) y comparar sus parámetros. **No demuestra que el sistema prediga compras o tendencias reales.**

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'scripts' / 'backtest.py').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))
import matplotlib.pyplot as plt
from scripts.backtest import crear_casos, evaluar, resumen, comparar
print('Proyecto:', ROOT)

## 1. Series de ejemplo

Creamos cinco series con un pico artificial y cinco sin pico, todas con 90 observaciones.

In [ ]:
casos = crear_casos()
fig, ax = plt.subplots(figsize=(11, 4))
for nombre, serie, viral in casos:
    ax.plot(serie.index, serie, alpha=0.65, label=nombre + (' · pico' if viral else ' · control'))
ax.axvline(casos[0][1].index[65], ls='--', color='black', alpha=0.5, label='Comienzo del pico simulado')
ax.set(title='Series temporales sintéticas de interés', xlabel='Fecha', ylabel='Índice artificial')
ax.legend(ncol=2, fontsize=7)
plt.show()

## 2. Resultados del detector

Definimos verdadero positivo cuando una serie con pico artificial recibe una alerta entre el día de inicio y su máximo. La anticipación es respecto al **máximo simulado**, no a ventas reales.

In [ ]:
resultados = evaluar(window=28, z_threshold=3.0, min_growth=0.5)
display(resultados)
print('Métricas:', resumen(resultados))

## 3. Comparación de parámetros

La comparación reutiliza los mismos ejemplos sintéticos. No sirve para seleccionar umbrales óptimos para el mundo real.

In [ ]:
comparacion = comparar()
display(comparacion)

## 4. Conclusión

El detector se puede ejecutar con resultados reproducibles y permite contrastar el número de alertas con la verdad conocida **que nosotros construimos**. Para completar una validación científica hay que reunir series de productos reales, fijar fechas de eventos y crear un conjunto de evaluación separado que no se use para ajustar los umbrales.

## 5. Observaciones reales de Wikipedia (opcional)

Esta sección consulta los 90 días de visitas de un artículo a la API local de TrendWatch. Requiere ejecutar Docker y realizar antes una ingesta con `SOURCE_MODE=wikipedia`. **Mide visitas a un artículo, no compras**.


In [ ]:
import requests
import pandas as pd
from trendwatch.detection.spike import find_spikes
url = "http://127.0.0.1:8000/trends/Taylor%20Swift"
try:
    respuesta = requests.get(url, params={"source": "wikipedia", "days": 90}, timeout=10)
    respuesta.raise_for_status()
    visitas = pd.DataFrame(respuesta.json()["points"])
except (requests.RequestException, ValueError, KeyError) as exc:
    visitas = pd.DataFrame()
    print("Ejecuta la API y la ingesta de Wikipedia para ver datos reales:", exc)
if len(visitas):
    visitas["date"] = pd.to_datetime(visitas["date"])
    visitas = visitas.set_index("date")
    ax = visitas["value"].plot(figsize=(10,4), title="Wikipedia · Taylor Swift · visitas diarias")
    ax.set(ylabel="Visitas al artículo", xlabel="Fecha")
    plt.show()
    real_spikes = find_spikes(visitas["value"], window=28)
    print("Alertas en la serie observada:", int(real_spikes["is_spike"].sum()))
